# Task 2
This serves as a template which will guide you through the implementation of this task. It is advised to first read the whole template and get a sense of the overall structure of the code before trying to fill in any of the TODO gaps.
This is the jupyter notebook version of the template. For the python file version, please refer to the file `template_solution.py`.

First, we import necessary libraries:

In [1]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import KFold
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import DotProduct, RBF, Matern, RationalQuadratic
from sklearn.metrics import r2_score
from sklearn.preprocessing import RobustScaler
from sklearn.preprocessing import MinMaxScaler
# Add any other imports you need here

# Data Loading
TODO: Perform data preprocessing, imputation and extract X_train, y_train and X_test
(and potentially change initialization of variables to accomodate how you deal with non-numeric data)

In [2]:
"""
This loads the training and test data, preprocesses it, removes the NaN
values and interpolates the missing data using imputation

Parameters
----------
Compute
----------
X_train: matrix of floats, training input with features
y_train: array of floats, training output with labels
X_test: matrix of floats: dim = (100, ?), test input with features
"""
# Load training data
original_train_df = pd.read_csv("train.csv") 
train_df = original_train_df.copy()

print("Original Training data:")
print(train_df.info())
print(train_df.head(2))

# Dummy initialization of the X_train, X_test and y_train   
# TODO: Depending on how you deal with the non-numeric data, you may want to 
# modify/ignore the initialization of these variables   

# TODO: Perform data preprocessing, imputation and extract X_train, y_train and X_test

Original Training data:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 900 entries, 0 to 899
Data columns (total 11 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   season     900 non-null    object 
 1   price_AUS  638 non-null    float64
 2   price_CHF  631 non-null    float64
 3   price_CZE  632 non-null    float64
 4   price_GER  631 non-null    float64
 5   price_ESP  631 non-null    float64
 6   price_FRA  636 non-null    float64
 7   price_UK   613 non-null    float64
 8   price_ITA  634 non-null    float64
 9   price_POL  635 non-null    float64
 10  price_SVK  619 non-null    float64
dtypes: float64(10), object(1)
memory usage: 77.5+ KB
None
   season  price_AUS  price_CHF  price_CZE  price_GER  price_ESP  price_FRA  \
0  spring        NaN   9.644028  -1.686248  -1.748076  -3.666005        NaN   
1  summer        NaN   7.246061  -2.132377  -2.054363  -3.295697  -4.104759   

   price_UK  price_ITA  price_POL  price_SVK  
0 -1.82

In [3]:
# Load test data
print("Original Test Data:")
test_df = pd.read_csv("test.csv")
print(test_df.info())
print(test_df.head(2))

Original Test Data:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 10 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   season     100 non-null    object 
 1   price_AUS  67 non-null     float64
 2   price_CZE  75 non-null     float64
 3   price_GER  69 non-null     float64
 4   price_ESP  61 non-null     float64
 5   price_FRA  75 non-null     float64
 6   price_UK   76 non-null     float64
 7   price_ITA  70 non-null     float64
 8   price_POL  65 non-null     float64
 9   price_SVK  66 non-null     float64
dtypes: float64(9), object(1)
memory usage: 7.9+ KB
None
   season  price_AUS  price_CZE  price_GER  price_ESP  price_FRA  price_UK  \
0  spring        NaN   0.472985   0.707957        NaN  -1.136441 -0.596703   
1  summer  -1.184837   0.358019        NaN  -3.199028  -1.069695       NaN   

   price_ITA  price_POL  price_SVK  
0        NaN   3.298693   1.921886  
1  -1.420091   3.238307       

#### Simple Imputer (passes easy baseline)

In [4]:
"""2 current options for imputation: Replace missing values with median or mean of the rest
I think the median would be better in this case as the distributions don't seem very Gaussian"""
"""
#print("Training data:")
#print(train_df.info())
#print(train_df.head(2))
imp_mean = SimpleImputer(strategy="median")
no_impute = train_df[["season","price_CHF"]]
to_impute = train_df.drop(columns=["season","price_CHF"])
imp_mean.fit(to_impute)

train_df = pd.concat([no_impute,pd.DataFrame(imp_mean.transform(to_impute), columns = to_impute.columns)],axis=1)
#print(train_df.info())
#print(train_df.head(2))

#print("Test data:")
#print(test_df.info())
#print(test_df.head(3))
no_impute = test_df[["season"]]
to_impute = test_df.drop(columns=["season"])
test_df = pd.concat([no_impute,pd.DataFrame(imp_mean.transform(to_impute), columns = to_impute.columns)],axis=1)
#print(test_df.info())
#print(test_df.head(3))
"""
print()

#### Iterative Imputer

In [5]:

#print("Training data:")
#print(train_df.info())
#print(train_df.head(3))
it_imp = IterativeImputer(initial_strategy="mean")
no_impute = train_df[["season","price_CHF"]]
to_impute = train_df.drop(columns=["season","price_CHF"])
it_imp.fit(to_impute)

train_df = pd.concat([no_impute,pd.DataFrame(it_imp.transform(to_impute), columns = to_impute.columns)],axis=1)
#print(train_df.info())
#print(train_df.head(3))

#print("Test data:")
#print(test_df.info())
#print(test_df.head(3))
no_impute = test_df[["season"]]
to_impute = test_df.drop(columns=["season"])
test_df = pd.concat([no_impute,pd.DataFrame(it_imp.transform(to_impute), columns = to_impute.columns)],axis=1)
#print(test_df.info())
#print(test_df.head(3))


In [6]:
"""drop all rows where price_CHF is missing bc that is what we are trying to learn"""
#print(train_df.info())
train_df = train_df.dropna(subset="price_CHF",ignore_index=True) #removes over 200 entries ; ignore_index=True resets the indexing
#print(train_df.info())


#### Categorical Attributes

In [7]:
"""Dealing with categorical attribute "season":
I don't think dropping this is viable as the season should HEAVILY effect the electricity pricing (e.g Switzerland even produces surplus electricity 
in the summer but has to import more in the winter)
Additionally, I don't think ordinal encoding (giving each a number) is viable as that implies an ordering which is a bit difficult
Therefore, I use one hot encoding (create 4 columns for each of the 4 present seasons with values 0 or 1)
"""
#I couldn't get the onehotencoder from sklearn to work so for now this has to be enough
#Note: We know that season ALWAYS has a value so it's fine if we don't handle NULLs

train_df = pd.get_dummies(train_df,dtype=int)
print(train_df.info())

test_df = pd.get_dummies(test_df,dtype=int)
print(test_df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 631 entries, 0 to 630
Data columns (total 14 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   price_CHF      631 non-null    float64
 1   price_AUS      631 non-null    float64
 2   price_CZE      631 non-null    float64
 3   price_GER      631 non-null    float64
 4   price_ESP      631 non-null    float64
 5   price_FRA      631 non-null    float64
 6   price_UK       631 non-null    float64
 7   price_ITA      631 non-null    float64
 8   price_POL      631 non-null    float64
 9   price_SVK      631 non-null    float64
 10  season_autumn  631 non-null    int32  
 11  season_spring  631 non-null    int32  
 12  season_summer  631 non-null    int32  
 13  season_winter  631 non-null    int32  
dtypes: float64(10), int32(4)
memory usage: 59.3 KB
None
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 13 columns):
 #   Column         Non-Null C

#### Scaling

In [8]:
"""scaler = MinMaxScaler()
tmp = train_df.drop(columns="price_CHF")
train_df = pd.concat([train_df['price_CHF'],pd.DataFrame(scaler.fit_transform(tmp),columns = tmp.columns)],axis=1)
test_df = pd.DataFrame(scaler.transform(test_df),columns = test_df.columns)
print(train_df.info())
print(train_df.head(3))
print(test_df.info())
print(test_df.head(3))"""
print()

In [9]:
X_train = train_df.drop(columns=['price_CHF'])
y_train = train_df['price_CHF']
print(X_train.info())
print(y_train.info())

X_test = test_df
#print(X_test.info())
assert (X_train.shape[1] == X_test.shape[1]) and (X_train.shape[0] == y_train.shape[0]) and (X_test.shape[0] == 100), "Invalid data shape"

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 631 entries, 0 to 630
Data columns (total 13 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   price_AUS      631 non-null    float64
 1   price_CZE      631 non-null    float64
 2   price_GER      631 non-null    float64
 3   price_ESP      631 non-null    float64
 4   price_FRA      631 non-null    float64
 5   price_UK       631 non-null    float64
 6   price_ITA      631 non-null    float64
 7   price_POL      631 non-null    float64
 8   price_SVK      631 non-null    float64
 9   season_autumn  631 non-null    int32  
 10  season_spring  631 non-null    int32  
 11  season_summer  631 non-null    int32  
 12  season_winter  631 non-null    int32  
dtypes: float64(9), int32(4)
memory usage: 54.4 KB
None
<class 'pandas.core.series.Series'>
RangeIndex: 631 entries, 0 to 630
Series name: price_CHF
Non-Null Count  Dtype  
--------------  -----  
631 non-null    float64
dtypes: float6

#### potential TO DOs: 
- other simple imputation strategies: (most_frequent,constant)
- more sophisticated imptutation
- scale features
- use some sort of feature transformations

Note: Cleaner Code can be written using the sklearn "Pipeline" but this works for now so

#### What did not work
- dropping any rows with missing data (at least one data point missing in each row)
- only picking certain columns (maybe I picked wrongly, but I think there's too little data to do that - as above most rows have most columns empty)

# Choosing a Model

In [10]:
""" Cross validation of course
Evaluate with r2 score, where 1.0 is best score"""
min_val = 1000
kf = KFold(n_splits=10,shuffle=True,random_state = 42)

kernels = {"DotProduct": DotProduct(), "RBF" : RBF(), "Matern" : Matern(), "RatQuad" : RationalQuadratic()}
""" try out dropping various columns
X_training_sets = [X_train, X_train.drop(columns=['price_UK','price_ESP']), #remove less strongly correlated ones \ 
                   X_train.drop(columns=['price_UK','price_ESP','price_CZE','price_AUS','price_FRA']), #remove more less correlated ones \  
                   X_train.drop(columns = ['price_CZE','price_ESP','price_FRA','price_UK','price_SVK'])] #only pick "not inter correlated" features and don't include less correlated ones
for i in range(4): 
-> did not work
"""
for name in kernels: 
    gpr = GaussianProcessRegressor(kernel=kernels[name])
    scores = cross_val_score(gpr,X_train,y_train,scoring="r2",cv=kf)
    print(scores)
    mean = np.mean(scores)
    print(name,":",mean)

[0.86423378 0.93412124 0.92574514 0.89479286 0.91257138 0.91463855
 0.91094812 0.91069222 0.89563577 0.91953321]
DotProduct : 0.9082912288629112


C:\Users\elena\anaconda3\Lib\site-packages\sklearn\gaussian_process\kernels.py:420: ConvergenceWarning: The optimal value found for dimension 0 of parameter length_scale is close to the specified lower bound 1e-05. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(
C:\Users\elena\anaconda3\Lib\site-packages\sklearn\gaussian_process\kernels.py:420: ConvergenceWarning: The optimal value found for dimension 0 of parameter length_scale is close to the specified lower bound 1e-05. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(
C:\Users\elena\anaconda3\Lib\site-packages\sklearn\gaussian_process\kernels.py:420: ConvergenceWarning: The optimal value found for dimension 0 of parameter length_scale is close to the specified lower bound 1e-05. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(
C:\Users\elena\anaconda3\Lib\site-packages\sklearn\gaussian_process\kernels.py:420: ConvergenceWarni

[ 0.74939226 -1.18408677 -1.96061711  0.8710479  -1.61714723 -1.54835559
 -1.87090348 -1.69281394  0.79556906 -2.65333425]
RBF : -1.0111249161742575
[0.93535444 0.96113442 0.93727185 0.95198808 0.96814554 0.97542217
 0.9583755  0.94541615 0.93704944 0.94368397]
Matern : 0.9513841551368584
[0.95784089 0.97161374 0.94773361 0.95893275 0.97969867 0.97419694
 0.96899897 0.95677598 0.95808519 0.96095214]
RatQuad : 0.9634828861248834


# Modeling and Prediction
TODO: Define the model and fit it using training data. Then, use test data to make predictions

In [11]:
"""
This defines the model, fits training data and then does the prediction
with the test data 

Parameters
----------
X_train: matrix of floats, training input with 10 features
y_train: array of floats, training output
X_test: matrix of floats: dim = (100, ?), test input with 10 features

Compute
----------
y_test: array of floats: dim = (100,), predictions on test set
"""

y_pred=np.zeros(X_test.shape[0])
#TODO: Define the model and fit it using training data. Then, use test data to make predictions
#Seems like above the RatQuad kernel got the best score so let's use that
gpr = GaussianProcessRegressor(kernel=RationalQuadratic())
gpr.fit(X_train,y_train)
y_pred = gpr.predict(X_test)
print(y_pred)
assert y_pred.shape == (100,), "Invalid data shape"

[-2.36024419e+00 -2.35988548e+00 -2.67933253e+00 -2.11228193e+00
 -1.86723341e+00 -1.07892367e+00  1.53963337e-01  3.48490506e-01
 -8.73147214e-04  1.42166653e+00  1.32033917e+00  1.87851256e+00
  2.43617313e+00  3.42317374e+00  3.01920322e+00  2.31341652e+00
  1.73828950e+00  1.49111232e+00  2.12267140e+00  1.58865516e+00
  3.75468512e+00  3.85101698e+00  3.38690034e+00  2.80765731e+00
  3.25695835e+00  4.21864988e+00  5.60440490e+00  7.66915411e+00
  9.12872345e+00  8.15322214e+00  8.47225318e+00  8.16178370e+00
  7.35465185e+00  7.08809134e+00  8.53037273e+00  7.78888111e+00
  7.48337593e+00  7.61592416e+00  7.52100745e+00  7.65366023e+00
  8.12866522e+00  7.93492920e+00  7.19587424e+00  8.17609659e+00
  7.91524386e+00  7.67668568e+00  7.58138554e+00  7.99045356e+00
  7.73977259e+00  7.77783276e+00  8.12734191e+00  9.02886099e+00
  8.90721870e+00  7.80465215e+00  7.77382510e+00  8.86989331e+00
  8.89961940e+00  8.33561880e+00  7.49499483e+00  6.77629959e+00
  6.79114554e+00  5.49471

# Saving Results
You don't have to change this

In [12]:
dt = pd.DataFrame(y_pred) 
dt.columns = ['price_CHF']
dt.to_csv('results.csv', index=False)
print("\nResults file successfully generated!")


Results file successfully generated!
